# Au–Pd reciprocal-space calibration for SED/4D-STEM

**Purpose.** Determine the reciprocal-space pixel scale and diffraction
distortion parameters from an Au–Pd standard scan.

**Input.** A HyperSpy-readable 4D-STEM dataset with acquisition voltage,
nominal camera length and convergence angle encoded in its filename, for
example `Au_300kV_145mm_0.5mrad.zspy`.

**Output.** A JSON record containing the reciprocal-space calibration,
fitted ellipse, accelerating voltage, convergence angle and nominal camera
length.

This notebook is a compact method record for experienced SED users. Raw
datasets are not distributed with the repository.

## 1. Setup and parameters

Edit only this section when adapting the workflow. GPU acceleration is
optional: when CuPy cannot detect a working CUDA device, Bragg-disk
detection automatically falls back to the CPU.

In [ ]:
%matplotlib inline

import json
import re
from pathlib import Path

import h5py
import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np
import py4DSTEM
from py4DSTEM.visualize import show

print("Python workflow: Au–Pd reciprocal calibration")
print("py4DSTEM:", py4DSTEM.__version__)
print("HyperSpy:", hs.__version__)
print("h5py:", h5py.__version__)

In [ ]:
# Repository-relative paths. This works when Jupyter starts from either
# the repository root or its notebooks/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_ROOT / "data/Au_300kV_145mm_0.5mrad.zspy"
OUTPUT_DIR = REPO_ROOT / "outputs/calibration"

# Execution
REQUEST_GPU = True
RANDOM_SEED = 0

# Data preparation and inspection
REAL_SPACE_REBIN = (1, 1, 1, 1)
HOT_PIXEL_THRESHOLD = 0.10
PROBE_SAMPLE_POSITION = (60, 50)
BRAGG_CHECK_POSITION = (80, 85)
ADF_RADII_PX = (60, 100)

# Synthetic probe and Bragg-disk detection
SYNTHETIC_PROBE_EDGE_WIDTH_PX = 10.0
N_TUNING_PATTERNS = 6
DETECTION_PARAMETERS = {
    "corrPower": 1.0,
    "sigma": 3.0,
    "edgeBoundary": 4,
    "minRelativeIntensity": 0.10,
    "minAbsoluteIntensity": 0.10,
    "minPeakSpacing": 3,
    "subpixel": "poly",
    "upsample_factor": 8,
    "maxNumPeaks": 1000,
}

# Au–Pd reference: random-substitutional FCC alloy
AU_FRACTION = 0.50
AU_LATTICE_A = 4.0782
PD_LATTICE_A = 3.8907
ELLIPSE_RING_HKL = (2, 2, 0)
ELLIPSE_RING_HALF_WIDTH_PX = 8
MAX_STRUCTURE_FACTOR_Q_INV_A = 1.5
PIXEL_SIZE_FIT_RANGE_INV_A = (0.0, 1.2)

# Optional manual initial value. Use None to infer it from voltage/camera length.
INITIAL_Q_PIXEL_SIZE_INV_A = None

In [ ]:
def detect_cuda():
    if not REQUEST_GPU:
        return False
    try:
        import cupy as cp

        available = cp.cuda.runtime.getDeviceCount() > 0
        if available:
            print("CUDA device detected: Bragg-disk detection will use the GPU.")
        return available
    except Exception as exc:
        print(f"CUDA unavailable ({type(exc).__name__}); using CPU.")
        return False


def bounded_position(position, navigation_shape):
    return tuple(
        min(max(int(value), 0), int(size) - 1)
        for value, size in zip(position, navigation_shape)
    )


def parse_acquisition_metadata(path):
    name = path.name
    voltage_match = re.search(r"(\d+)kv", name, re.IGNORECASE)
    convergence_match = re.search(r"(\d+(?:\.\d+)?)mrad", name, re.IGNORECASE)
    camera_match = re.search(r"(\d+)mm", name, re.IGNORECASE)

    if not (voltage_match and convergence_match and camera_match):
        raise ValueError(
            "Filename must include voltage, camera length and convergence "
            "angle, e.g. Au_300kV_145mm_0.5mrad.zspy."
        )

    return {
        "accelerating_voltage_v": float(voltage_match.group(1)) * 1_000,
        "convergence_angle_mrad": float(convergence_match.group(1)),
        "nominal_camera_length_m": float(camera_match.group(1)) / 1_000,
    }


def estimate_initial_q_pixel_size(voltage_v, camera_length_m):
    # Empirical starting values used for Cambridge Merlin acquisitions.
    lookup = {
        (300_000, 0.058): 0.0296,
        (300_000, 0.115): 0.0129,
        (300_000, 0.145): 0.0103,
        (300_000, 0.185): 0.0080,
        (300_000, 0.230): 0.0072,
        (300_000, 0.200): 0.0093,
        (300_000, 0.400): 0.0049,
        (300_000, 0.800): 0.0024,
        (200_000, 0.090): 0.0110,
        (200_000, 0.200): 0.0053,
        (200_000, 0.400): 0.0027,
        (200_000, 0.800): 0.0014,
        (200_000, 1.000): 0.0009,
        (80_000, 2.000): 0.0053,
    }
    key = (int(voltage_v), round(float(camera_length_m), 3))
    if key not in lookup:
        raise ValueError(
            f"No initial reciprocal calibration is registered for {key}. "
            "Set INITIAL_Q_PIXEL_SIZE_INV_A manually."
        )
    return lookup[key]


USE_CUDA = detect_cuda()
ACQUISITION = parse_acquisition_metadata(DATA_PATH)
INITIAL_Q = (
    INITIAL_Q_PIXEL_SIZE_INV_A
    if INITIAL_Q_PIXEL_SIZE_INV_A is not None
    else estimate_initial_q_pixel_size(
        ACQUISITION["accelerating_voltage_v"],
        ACQUISITION["nominal_camera_length_m"],
    )
)

print(ACQUISITION)
print("Initial reciprocal pixel size (Å⁻¹ px⁻¹):", INITIAL_Q)

## 2. Load and prepare the standard scan

The HyperSpy signal is passed to a py4DSTEM `DataCube`. Hot pixels are
removed before diffraction statistics and Bragg-disk detection.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Update DATA_PATH in the parameter cell."
    )

signal = hs.load(DATA_PATH)
if REAL_SPACE_REBIN != (1, 1, 1, 1):
    signal = signal.rebin(scale=REAL_SPACE_REBIN)

datacube = py4DSTEM.datacube.DataCube(data=signal.data)
datacube.calibration.set_Q_pixel_size(INITIAL_Q)
datacube.calibration.set_Q_pixel_units("A^-1")

datacube, hot_pixel_mask = datacube.filter_hot_pixels(
    thresh=HOT_PIXEL_THRESHOLD,
    return_mask=True,
)
print("Data shape:", datacube.data.shape)
print("Hot pixels removed:", int(np.count_nonzero(hot_pixel_mask)))

In [ ]:
datacube.get_dp_mean()
datacube.get_dp_max()

py4DSTEM.show(
    [datacube.tree("dp_mean"), datacube.tree("dp_max")],
    cmap="inferno",
    power=0.5,
    figsize=(6, 3),
)

## 3. Estimate the probe and select tuning patterns

A synthetic probe kernel is used for disk correlation. Tuning positions
are sampled reproducibly from pixels above the mean virtual dark-field
intensity.

In [ ]:
navigation_shape = datacube.data.shape[:2]
probe_position = bounded_position(PROBE_SAMPLE_POSITION, navigation_shape)
probe_pattern = datacube[probe_position]

probe_semiangle, probe_qx0, probe_qy0 = (
    py4DSTEM.process.calibration.get_probe_size(probe_pattern)
)
diffraction_center = (probe_qx0, probe_qy0)
print("Probe radius and centre (px):", probe_semiangle, diffraction_center)

datacube.get_virtual_image(
    mode="annulus",
    geometry=(diffraction_center, ADF_RADII_PX),
    name="dark_field",
)
show(datacube.tree("dark_field"))

In [ ]:
synthetic_probe = py4DSTEM.braggvectors.probe.Probe.generate_synthetic_probe(
    float(probe_semiangle),
    SYNTHETIC_PROBE_EDGE_WIDTH_PX,
    datacube.data.shape[-2:],
)
probe_kernel = synthetic_probe.get_kernel(
    mode="sigmoid",
    radii=(float(probe_semiangle) * 0.5, float(probe_semiangle) * 3.0),
    bilinear=True,
)

dark_field = datacube.tree("dark_field").data
candidate_positions = np.column_stack(np.where(dark_field > dark_field.mean()))
if candidate_positions.size == 0:
    raise RuntimeError("No tuning positions were found above the mean dark-field intensity.")

rng = np.random.default_rng(RANDOM_SEED)
selected = candidate_positions[
    rng.choice(
        len(candidate_positions),
        size=min(N_TUNING_PATTERNS, len(candidate_positions)),
        replace=False,
    )
]
tuning_x = tuple(selected[:, 0].astype(int))
tuning_y = tuple(selected[:, 1].astype(int))

py4DSTEM.visualize.show_points(
    datacube.tree("dark_field"),
    x=selected[:, 0],
    y=selected[:, 1],
    figsize=(6, 6),
)

In [ ]:
tuning_disks = datacube.find_Bragg_disks(
    data=(tuning_x, tuning_y),
    template=probe_kernel,
    **DETECTION_PARAMETERS,
)

py4DSTEM.visualize.show_image_grid(
    get_ar=lambda i: datacube.data[tuning_x[i], tuning_y[i]],
    H=int(np.ceil(len(tuning_x) / 2)),
    W=2,
    axsize=(4, 4),
    power=0.2,
    get_x=lambda i: tuning_disks[i].data["qx"],
    get_y=lambda i: tuning_disks[i].data["qy"],
    open_circles=True,
    scale=500,
)

## 4. Detect Bragg disks and correct the diffraction origin

`USE_CUDA` is `False` when no compatible GPU is available; py4DSTEM then
executes the same disk search on the CPU.

In [ ]:
bragg_peaks = datacube.find_Bragg_disks(
    template=probe_kernel,
    CUDA=USE_CUDA,
    CUDA_batched=USE_CUDA,
    **DETECTION_PARAMETERS,
)

check_position = bounded_position(BRAGG_CHECK_POSITION, navigation_shape)
check_pattern = datacube[check_position]
check_vectors = bragg_peaks.raw[check_position]
show(
    check_pattern,
    power=0.5,
    points={"x": check_vectors.qx, "y": check_vectors.qy},
)

In [ ]:
raw_bragg_map = bragg_peaks.histogram(mode="raw", sampling=1)
qx0_measured, qy0_measured, origin_mask = bragg_peaks.measure_origin(
    center_guess=diffraction_center,
    score_method="intensity",
)
qx0_fit, qy0_fit, qx_residuals, qy_residuals = bragg_peaks.fit_origin(
    robust=True,
    robust_thresh=1.2,
)

centred_bragg_map = bragg_peaks.histogram(mode="cal", sampling=1)
show([raw_bragg_map, centred_bragg_map], scaling="log", vmax=0.999)

## 5. Fit elliptical distortion

The fit uses an isolated FCC reference ring predicted from the nominal
Au–Pd alloy lattice and the initial reciprocal calibration.

In [ ]:
pd_fraction = 1.0 - AU_FRACTION
alloy_lattice_a = AU_LATTICE_A * AU_FRACTION + PD_LATTICE_A * pd_fraction

hkl_norm = np.linalg.norm(np.asarray(ELLIPSE_RING_HKL, dtype=float))
reference_ring_q = hkl_norm / alloy_lattice_a
ring_radius_px = int(round(reference_ring_q / INITIAL_Q))
ellipse_fit_range = (
    ring_radius_px - ELLIPSE_RING_HALF_WIDTH_PX,
    ring_radius_px + ELLIPSE_RING_HALF_WIDTH_PX,
)

show(
    centred_bragg_map,
    cmap="gray",
    annulus={
        "center": centred_bragg_map.origin,
        "radii": ellipse_fit_range,
        "fill": True,
        "color": "r",
        "alpha": 0.3,
    },
)

ellipse_parameters = py4DSTEM.process.calibration.fit_ellipse_1D(
    centred_bragg_map,
    center=centred_bragg_map.origin,
    fitradii=ellipse_fit_range,
)
py4DSTEM.visualize.show_elliptical_fit(
    centred_bragg_map,
    ellipse_fit_range,
    ellipse_parameters,
    cmap="gray",
)

bragg_peaks.calibration.set_p_ellipse(ellipse_parameters)
bragg_peaks.setcal()
print("Ellipse parameters:", ellipse_parameters)
print("Ellipse aspect ratio:", ellipse_parameters[2] / ellipse_parameters[3])

## 6. Refine the reciprocal-space pixel scale

The standard is represented as a random-substitutional FCC alloy. Au and
Pd occupy the same lattice sites with occupancies set in the parameter
cell; the lattice parameter is estimated using Vegard's law.

In [ ]:
fcc_positions = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.0, 0.5, 0.5],
        [0.5, 0.0, 0.5],
        [0.5, 0.5, 0.0],
    ]
)
atomic_numbers = np.array([79, 46])
site_occupancies = np.array([AU_FRACTION, pd_fraction])

crystal = py4DSTEM.process.diffraction.Crystal(
    np.repeat(fcc_positions, len(atomic_numbers), axis=0),
    np.tile(atomic_numbers, len(fcc_positions)),
    alloy_lattice_a,
    occupancy=np.tile(site_occupancies, len(fcc_positions)),
)
crystal.calculate_structure_factors(MAX_STRUCTURE_FACTOR_Q_INV_A)

crystal.calibrate_pixel_size(
    bragg_peaks=bragg_peaks,
    bragg_k_power=2.0,
    plot_result=True,
    k_min=PIXEL_SIZE_FIT_RANGE_INV_A[0],
    k_max=PIXEL_SIZE_FIT_RANGE_INV_A[1],
    set_calibration_in_place=True,
)

calibrated_q_pixel_size = float(
    bragg_peaks.calibration["Q_pixel_size"]
)
print("Calibrated reciprocal pixel size (Å⁻¹ px⁻¹):", calibrated_q_pixel_size)

## 7. Export the calibration

The output directory is created if needed. No existing directory is
cleared or recursively deleted.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
calibration_record = {
    "source_file": DATA_PATH.name,
    "accelerating_voltage_v": ACQUISITION["accelerating_voltage_v"],
    "convergence_angle_mrad": ACQUISITION["convergence_angle_mrad"],
    "nominal_camera_length_m": ACQUISITION["nominal_camera_length_m"],
    "reciprocal_space_pixel_size_inv_a": calibrated_q_pixel_size,
    "ellipse_parameters": [float(value) for value in ellipse_parameters],
    "au_fraction": AU_FRACTION,
    "pd_fraction": pd_fraction,
    "alloy_lattice_parameter_a": float(alloy_lattice_a),
    "used_cuda": bool(USE_CUDA),
}

calibration_path = OUTPUT_DIR / f"{DATA_PATH.stem}_calibration.json"
with calibration_path.open("w", encoding="utf-8") as stream:
    json.dump(calibration_record, stream, indent=2)

print("Calibration written to:", calibration_path)
calibration_record

## Interpretation

Inspect the disk overlays, centred Bragg-vector map and ellipse fit before
accepting the exported values. The final scale and distortion parameters
are specific to the microscope voltage, nominal camera length, detector
geometry and acquisition configuration used for the standard scan.